# FisheriesAudit ALG 2026 — Entrega #01
## El Triángulo de Auditoría Pesquera: CBA · CMP · Captura Real

**Autor:** Ariel L. Giamportone  
**Filiación:** Ingeniero Pesquero | Docente Investigador | Data Scientist  
**Serie:** FisheriesAudit ALG 2026 — Gobernanza Pesquera Argentina  
**Fecha:** 2026-05-31

---

### Resumen

Este análisis cuantifica la brecha entre la Captura Biológicamente Aceptable (CBA) 
recomendada por el INIDEP y la Captura Máxima Permisible (CMP) aprobada por el 
Consejo Federal Pesquero (CFP), contrastándola con las capturas reales declaradas 
ante la SAGPyA. El conjunto de estos tres indicadores constituye el **Triángulo de 
Auditoría Pesquera**, una metodología original para evaluar la coherencia entre la 
gestión pesquera y la evidencia científica disponible.

**Palabras clave:** gobernanza pesquera, cuotas de pesca, INIDEP, CFP, Argentina, 
auditoría de datos, sostenibilidad

---

### Marco regulatorio

> La Ley 24.922 (Régimen Federal de Pesca) establece en su Art. 9 que el CFP 
> determinará las Capturas Máximas Permisibles "teniendo en cuenta las 
> recomendaciones científicas disponibles" del INIDEP. Este trabajo evalúa 
> empíricamente el grado de cumplimiento de ese mandato legal.

In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path(".").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src.analysis.inidep_comparator import INIDEPComparator
from src.analysis.research_exporter import ResearchExporter, SERIES_BRAND
from src.analysis.linkedin_formatter import LinkedInFormatter

DB_PATH = Path("data/processed/catalog.db")
OUT_DIR = Path("outputs/FisheriesAudit_ALG")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figuras").mkdir(exist_ok=True)

print("✓ Dependencias cargadas")
print(f"  pandas {pd.__version__} | scipy | matplotlib {plt.matplotlib.__version__}")

## 1. Carga de datos

Los datos provienen de tres fuentes públicas argentinas:

| Fuente | Descripción | URL |
|--------|-------------|-----|
| **INIDEP** | Informes Técnicos Operativos (ITO) — CBA por especie/año | marabierto.inidep.edu.ar |
| **CFP** | Actas públicas — CMP aprobada en sesión | cfp.gob.ar |
| **SAGPyA/SIPA** | Sistema Integrado de Pesca Argentina — capturas declaradas | sipa.magyp.gob.ar |

In [ ]:
comp = INIDEPComparator(DB_PATH)
comp.compute_comparisons()

df = comp.get_triangulo_completo()

print(f"Registros en triángulo de auditoría: {len(df)}")
print(f"Especies: {sorted(df['especie'].unique())}")
print(f"Años: {sorted(df['year'].unique())}")
df.head(10)

## 2. Estadística descriptiva

Distribución de los ratios CMP/CBA (sobreasignación) y Captura/CBA.

In [ ]:
df_r = df.dropna(subset=["ratio_cmp_cba"])
desc = df_r["ratio_cmp_cba"].describe()

print("=== Ratio CMP/CBA (cuota CFP / recomendación INIDEP) ===")
print(f"  n          : {int(desc['count'])}")
print(f"  Media      : {desc['mean']:.3f}")
print(f"  Mediana    : {df_r['ratio_cmp_cba'].median():.3f}")
print(f"  Desv. est. : {desc['std']:.3f}")
print(f"  % sobre CBA: {(df_r['ratio_cmp_cba'] > 1.0).mean()*100:.1f}%")

if "ratio_captura_cba" in df.columns:
    df_c = df.dropna(subset=["ratio_captura_cba"])
    print()
    print("=== Ratio Captura/CBA (captura real / recomendación INIDEP) ===")
    print(f"  n          : {len(df_c)}")
    print(f"  Mediana    : {df_c['ratio_captura_cba'].median():.3f}")
    print(f"  % sub-util (<70% CMP): {(df_c['ratio_captura_cba'] < 0.7).mean()*100:.1f}%")

## 3. Triángulo de auditoría — Figura 1

**Figura 1.** Triángulo de Auditoría Pesquera: CBA (INIDEP), CMP (CFP) y Captura real (SAGPyA).

In [ ]:
exp = ResearchExporter(comp, output_dir=OUT_DIR)

especies = df["especie_code"].dropna().unique()
for esp_code in sorted(especies):
    try:
        fig = exp.figura_triangulo_por_especie(esp_code, save=True)
        plt.show()
        print(f"  ✓ {esp_code} → {OUT_DIR}/figuras/triangulo_{esp_code}.png")
    except Exception as e:
        print(f"  ✗ {esp_code}: {e}")

### Figura 2. Diagnóstico comparativo — Todas las especies

Cuadrante superior derecho = sobreasignación + sobrepesca simultáneas (zona crítica).

In [ ]:
fig = exp.figura_comparacion_todas_especies(save=True)
plt.show()

## 4. Tests estadísticos

Tests no paramétricos (sin asunción de normalidad):

| Test | H₀ | α |
|------|-----|---|
| Wilcoxon signed-rank | mediana CMP/CBA = 1 | 0.05 |
| Kendall τ | sin tendencia temporal | 0.05 |
| Kruskal-Wallis | mismo ratio en todas las especies | 0.05 |
| Spearman ρ | sin correlación CMP/CBA ↔ Captura/CBA | 0.05 |

In [ ]:
tests = exp.ejecutar_todos_los_tests()
df_tests = pd.DataFrame([t.to_dict() for t in tests])

for t in tests:
    sig = "*** p<0.05" if t.significativo else f"    p={t.p_value:.4f}"
    print(f"[{sig:14s}] {t.nombre}")
    print(f"               {t.interpretacion}")
    print()

## 5. Hallazgos estructurados

In [ ]:
hallazgos = exp.generar_hallazgos()

for i, h in enumerate(hallazgos, 1):
    print(f"{'='*60}")
    print(f"Hallazgo {i} [{h.nivel_evidencia.upper()}]: {h.titulo}")
    print(h.descripcion)
    print("Datos clave:")
    for k, v in h.datos_clave.items():
        print(f"  · {k}: {v}")
    if h.tests:
        for t in h.tests:
            print(f"  · {t.nombre}: p={t.p_value:.4f} ({'sig.' if t.significativo else 'no sig.'})")
    print()

## 6. Exportación de datasets (FAIR)

In [ ]:
csv_path = exp.exportar_csv()
print(f"CSV: {csv_path}")

excel_path = exp.exportar_excel()
print(f"Excel: {excel_path}")

latex = exp.exportar_tabla_latex()
print()
print("LaTeX (primeras líneas):")
print(latex[:300])

## 7. Posts LinkedIn — Serie FisheriesAudit ALG

In [ ]:
formatter = LinkedInFormatter(hallazgos, numero_entrega_inicial=1)
todos = formatter.generar_todos()

# Entrega #01 — perfil personal
print("=== PERFIL PERSONAL ===")
print(todos["personal"][0].render())

In [ ]:
# Entrega #01 — Pesqueros en IA
print("=== PESQUEROS EN IA ===")
print(todos["pesqueros_ia"][1].render())

## 8. Metodología y reproducibilidad

### Datos

- **INIDEP**: ITOs disponibles en marabierto.inidep.edu.ar. Datos seed verificados para 5 especies, 2019–2025.
- **CFP**: Actas públicas de sesión, cfp.gob.ar. Procesadas mediante extracción PDF + NER pesquero especializado.
- **SAGPyA/SIPA**: Capturas declaradas por especie y año, acceso público.

### Reproducibilidad

```bash
git clone https://github.com/arielgiamportone/cfp-audit-intelligence
pip install -r requirements.txt
python scripts/run_full_pipeline.py --step inidep
jupyter notebook notebooks/FisheriesAudit_ALG_01_triangulo_auditoria.ipynb
```

### Limitaciones

- Datos seed: 5 especies, ~40 registros verificados manualmente.
- El pipeline completo requiere ejecutar el scraper sobre las 492+ actas CFP.
- Las capturas SAGPyA son datos declarados por las empresas pesqueras.

### Declaración de conflicto de intereses

El autor no tiene vínculos económicos con empresas pesqueras ni con organismos reguladores.
El análisis es descriptivo y no constituye acusación legal.

---

*FisheriesAudit ALG 2026 — Ariel L. Giamportone*  
*Ing. Pesquero | Docente Investigador | Data Scientist*